# Diabetic Retinopathy Severity Classification

This cleaned notebook reproduces the documented five-class CNN workflow. Download the source dataset and run `src/prepare_data.py` before executing the cells. Dataset images and original notebook outputs are intentionally excluded from the public repository.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

from model import CLASS_NAMES, build_model
from train import collect_samples, make_dataset, split_samples

tf.keras.utils.set_random_seed(0)

## Data split

The prepared dataset contains 700 images for each class. The split is stratified at 70% training, 15% validation, and 15% testing.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
BATCH_SIZE = 32

paths, labels = collect_samples(DATA_DIR)
train_paths, valid_paths, test_paths, train_labels, valid_labels, test_labels = split_samples(
    paths, labels, seed=0
)

print({
    'train': len(train_paths),
    'validation': len(valid_paths),
    'test': len(test_paths),
})

In [ ]:
train_data = make_dataset(train_paths, train_labels, BATCH_SIZE, shuffle=True, seed=0)
valid_data = make_dataset(valid_paths, valid_labels, BATCH_SIZE, shuffle=False, seed=0)
test_data = make_dataset(test_paths, test_labels, BATCH_SIZE, shuffle=False, seed=0)

## Model training

The compact CNN has three convolutional blocks, global average pooling, a 125-unit dense layer, dropout, and a five-class softmax output.

In [ ]:
model = build_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=8, restore_best_weights=True
)
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=50,
    callbacks=[early_stopping],
)

## Evaluation

In [ ]:
probabilities = model.predict(test_data, verbose=0)
predictions = probabilities.argmax(axis=1)
print(classification_report(
    test_labels,
    predictions,
    labels=list(CLASS_NAMES),
    target_names=list(CLASS_NAMES.values()),
    zero_division=0,
))
print(confusion_matrix(test_labels, predictions, labels=list(CLASS_NAMES)))

## Recorded run

The original documented run stopped after 28 epochs, reached its best validation accuracy of **39.05%** at epoch 20, and produced **38.48% test accuracy**, **42.38% macro precision**, **38.48% macro recall**, and **33.94% macro F1** on 525 balanced test images. Re-running can produce different values across hardware and TensorFlow versions.